<div style="padding:30px;border-radius:20px;background:linear-gradient(120deg,#0f172a,#1c5cab 55%,#2a78d6);color:white">
<div style="font-size:12px;letter-spacing:.14em;opacity:.86">POK&Eacute;MON TCG AI BATTLE &middot; ALAKAZAM &middot; POWERFUL HAND</div>
<h1 style="margin:8px 0;color:white">PTCG 1071 Alakazam &mdash; the rule-based skeleton</h1>
<div style="font-size:17px;opacity:.95">The decklist and the hand-written policy that sit underneath a submission whose game history peaked at 1071 and settled at a public score of 890.2.</div>
</div>


## What this notebook contains

Powerful Hand deals **20 damage for every card in your hand**, so an Alakazam deck is really a
draw engine with an attack bolted on. Playing it well is mostly about *when not to draw*.

This notebook publishes the part of my agent that answers that question:

- the **60-card decklist**, and
- the **rule-based skeleton policy** &mdash; a single `main.py` that decides every prompt on its own.

The skeleton is a complete, valid submission: the notebook materialises it, builds
`submission.tar.gz` against the official engine, and runs a loader-faithful smoke test on it.

**What it does not contain.** The agent that recorded the rating below runs three more layers on
top of this skeleton (in-turn lethal search, a turn-planning rollout, and a learned leaf
evaluator). Those layers are described in words further down, but their code is not included here.
So: the archive this notebook builds is *the skeleton alone*, not the rated agent.


In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import os
import shutil
import sys
import tarfile

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
print('working directory:', WORK)


## 1. The deck

| line | why it is here |
|---|---|
| Abra / Kadabra / Alakazam &times;4 | Powerful Hand (20 &times; cards in hand). Alakazam's Psychic Draw refills to 6 every turn. |
| Dunsparce / Dudunsparce | Run Away Draw is a second engine; Dunsparce's Dig blocks all effects on a heads flip. |
| Fezandipiti ex &times;1 | Flip the Script &mdash; three cards after a KO, i.e. +60 damage on the next swing. |
| Rare Candy &times;4 | Skips Kadabra so the attacker lands a turn earlier. |
| Buddy-Buddy Poffin / Poke Pad &times;4 | Setup search. Both burn deck, which is what the skeleton spends most of its logic managing. |
| Enhanced Hammer &times;4 / Xerosic's Machinations &times;2 | Mist Energy blanks Powerful Hand outright, so removal is mandatory, not optional. |
| Boss's Orders &times;2 | Pull the engine Pokemon, not the biggest one. |
| Night Stretcher &times;3 / Lana's Aid &times;1 | Recover energy and refill the deck in long games. |

Hand size is damage, so every optional draw has a real cost: it converts deck into hand *now*
but shortens the game. The whole skeleton is built around that trade.


In [ ]:
from collections import OrderedDict

DECK_TABLE = [[741, "Abra", "Pokemon", 4], [742, "Kadabra", "Pokemon", 4], [743, "Alakazam", "Pokemon", 4], [65, "Dunsparce", "Pokemon", 4], [66, "Dudunsparce", "Pokemon", 3], [140, "Fezandipiti ex", "Pokemon", 1], [1086, "Buddy-Buddy Poffin", "Item", 4], [1152, "Poke Pad", "Item", 4], [1079, "Rare Candy", "Item", 4], [1081, "Enhanced Hammer", "Item", 4], [1097, "Night Stretcher", "Item", 3], [1225, "Hilda", "Supporter", 4], [1231, "Dawn", "Supporter", 4], [1182, "Boss's Orders", "Supporter", 2], [1197, "Xerosic's Machinations", "Supporter", 2], [1184, "Lana's Aid", "Supporter", 1], [5, "Basic Psychic Energy", "Basic energy", 3], [19, "Telepath Psychic Energy", "Special energy", 4], [13, "Enriching Energy", "Special energy", 1]]

composition = OrderedDict()
for card_id, card, category, count in DECK_TABLE:
    composition[category] = composition.get(category, 0) + count

assert sum(composition.values()) == 60, composition
print('%-16s %s' % ('category', 'cards'))
for category, count in composition.items():
    print('%-16s %5d' % (category, count))
print('%-16s %5d' % ('total', sum(composition.values())))


## 2. How the skeleton plays

The policy scores every option the engine offers and takes the best one. The scores encode a
handful of rules that all come from the same observation &mdash; *cards in hand are damage, and the
deck is a finite resource*:

**Draw discipline**
- **Over-draw brake.** Sixteen cards in hand is 320 damage, enough to one-shot anything on the
  board. Past that, optional draw only burns deck, so it is declined.
- **Hand preservation.** On a developed board a marginal optional play is worth less than the
  +20 damage of simply holding the card. Removing Mist Energy, setting up a KO, mandatory
  development and thin-deck recovery are the exceptions that always fire.
- **Thin-deck floor.** Below a threshold the draw and search abilities are declined outright &mdash;
  but only in mill / stall contexts, where deck-out is the actual loss condition. Applying that
  floor unconditionally would throttle our own damage in a race.
- **Whiff guard.** Before playing a search card, count every visible copy of its targets &mdash; hand,
  discard, field, and the cards sitting under evolutions. If the deck and prizes provably hold
  none, the card is not played.

**Attacking**
- **Effective damage, not printed damage.** Powerful Hand places damage counters, which is an
  effect, so it is fully nullified while the defender carries Mist Energy. Under Mist the policy
  drops Powerful Hand's score and prefers a true damage attack or a switch.
- **Threat list.** Engine Pokemon (Munkidori, the Dragapult ex line) are worth removing ahead of
  raw prize value.
- **Attack lock.** Energy is attached but the engine offers no attack &mdash; something sealed it.
  Retreating the active clears the lock.

**Staying alive**
- **Empty field is an instant loss**, so with one or fewer Pokemon in play an immediately playable
  Basic outranks everything, including energy recovery.
- **One-shot pressure.** If the opponent can already deal 100+, a fragile evolution base is never
  promoted into the active spot.
- **Energy drought / Hammer insurance.** If the attacker is stranded without Psychic energy,
  refuelling outranks further development; once the opponent is seen using Enhanced Hammer, the
  main attacker is fed basic energy instead of special.


## 3. Where this sits in the full stack

The rated agent is four layers. Only the bottom one is in this notebook.

| layer | what it does | included here |
|---|---|---|
| Opponent deck inference | matches visible cards against known meta lists to fill in hidden information | described only |
| Leaf evaluator | hand-built features + a small MLP ensemble, scoring a position for the rollout | described only |
| Turn planning | expands the skeleton's top-scoring candidate moves and rolls them out, one to two plies | described only |
| **Rule-based skeleton** | **scores every prompt directly; decides the whole game on its own** | **yes &mdash; this notebook** |

The layering matters more than any single layer: the search only ever explores moves the skeleton
already ranks highly, so a better skeleton is also a better search. That is why the skeleton is
the part worth reading.


## 4. Materialize the skeleton

In [ ]:
# The skeleton, embedded verbatim. The hashes are checked before anything is written.
PAYLOADS = {
    "main.py": "...",
    "deck.csv": "NQ0KNQ0KNQ0KMTMNCjE5DQoxOQ0KMTkNCjE5DQo2NQ0KNjUNCjY1DQo2NQ0KNjYNCjY2DQo2Ng0KNzQxDQo3NDENCjc0MQ0KNzQxDQo3NDINCjc0Mg0KNzQyDQo3NDINCjc0Mw0KNzQzDQo3NDMNCjc0Mw0KMTA3OQ0KMTA3OQ0KMTA3OQ0KMTA3OQ0KMTA4MQ0KMTA4MQ0KMTA4MQ0KMTA4MQ0KMTA4Ng0KMTA4Ng0KMTA4Ng0KMTA4Ng0KMTA5Nw0KMTA5Nw0KMTA5Nw0KMTQwDQoxMTUyDQoxMTUyDQoxMTUyDQoxMTUyDQoxMTgyDQoxMTgyDQoxMTg0DQoxMTk3DQoxMTk3DQoxMjI1DQoxMjI1DQoxMjI1DQoxMjI1DQoxMjMxDQoxMjMxDQoxMjMxDQoxMjMxDQo=",
}
EXPECTED_SHA256 = {
    "main.py": "9b9f71473f7997812a8cc71d4aff7ea1ed0f0b1f6db00bd70704139939219b15",
    "deck.csv": "ecc0c329107627ae310b88f62658f1ae8dcc0dcd0084d6d94d8b60508b2d725f",
}

for name, blob in PAYLOADS.items():
    data = base64.b64decode(blob)
    digest = hashlib.sha256(data).hexdigest()
    assert digest == EXPECTED_SHA256[name], (name, digest)
    (WORK / name).write_bytes(data)
    print('%-9s %7d bytes  sha256 %s' % (name, len(data), digest))


## 5. Build a competition archive against the official engine

In [ ]:
def find_cg_source() -> Path:
    env_path = os.environ.get('PTCG_CG_DIR')
    candidates = [Path(env_path)] if env_path else []
    candidates += [
        Path('/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/cg'),
        Path('/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/sample_submission/cg'),
        Path('/kaggle/input/pokemon-tcg-ai-battle/sample_submission/cg'),
        Path('/kaggle/input/pokemon-tcg-ai-battle/sample_submission/sample_submission/cg'),
        Path.cwd() / 'cg',
    ]
    for candidate in candidates:
        if (candidate / 'api.py').is_file():
            return candidate
    for root in [Path('/kaggle/input'), Path.cwd()]:
        if root.is_dir():
            for found in root.rglob('cg/api.py'):
                return found.parent
    raise FileNotFoundError('official cg package not found')


cg_dir = find_cg_source()
print('engine package:', cg_dir)

build = WORK / 'skeleton_build'
if build.exists():
    shutil.rmtree(build)
build.mkdir()
shutil.copy2(WORK / 'main.py', build / 'main.py')
shutil.copy2(WORK / 'deck.csv', build / 'deck.csv')
shutil.copytree(cg_dir, build / 'cg')

archive = WORK / 'submission.tar.gz'
if archive.exists():
    archive.unlink()
with tarfile.open(archive, 'w:gz') as tar:
    for member in sorted(build.iterdir(), key=lambda p: p.name):
        tar.add(member, arcname=member.name)

deck_ids = [int(x) for x in (build / 'deck.csv').read_text().split()]
print('deck entries:', len(deck_ids), '/ distinct card ids:', len(set(deck_ids)))
print('archive:', archive, '%.2f MB' % (archive.stat().st_size / 1e6))


## 6. Loader-faithful smoke test

In [ ]:
# The competition loader takes the LAST callable defined by main.py, with no __file__ available.
# Reproduce that exactly, then probe the entry point with degenerate observations.
old_cwd = Path.cwd()
sys.path.insert(0, str(build))
os.chdir(build)
try:
    namespace = {'__name__': '__main__'}
    exec(compile((build / 'main.py').read_text(encoding='utf-8'), 'main.py', 'exec'), namespace)
    callables = [name for name, value in namespace.items() if callable(value)]
    loaded = [value for value in namespace.values() if callable(value)][-1]
    print('callable tail:', callables[-5:])
    assert getattr(loaded, '__name__', '') == 'agent', loaded

    returned_deck = loaded({'select': None})
    assert len(returned_deck) == 60, len(returned_deck)
    print('deck request returns', len(returned_deck), 'card ids')

    for label, probe in [('empty', {}),
                         ('null-select', {'current': None, 'select': None}),
                         ('partial', {'current': None})]:
        result = loaded(probe)
        assert isinstance(result, list), (label, type(result))
        print('probe %-12s -> %r' % (label, result))
finally:
    os.chdir(old_cwd)
    sys.path.remove(str(build))
print('loader-faithful smoke test passed')


## 7. Figures

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Patch

try:
    from IPython.display import Image, display
except ImportError:  # outside a notebook the figures are only written to disk
    Image = None
    display = print

PAL = {'surface': '#fcfcfb', 'ink': '#0b0b0b', 'ink2': '#52514e', 'muted': '#898781',
       'grid': '#e1e0d9', 'axis': '#c3c2b7', 'series': '#2a78d6', 'inactive': '#dcdbd4'}
plt.rcParams.update({'font.family': 'sans-serif', 'figure.facecolor': PAL['surface'],
                     'axes.facecolor': PAL['surface'], 'savefig.facecolor': PAL['surface']})
VIS = WORK / 'skeleton_visuals'
VIS.mkdir(exist_ok=True)


def show(fig, name):
    path = VIS / name
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.close(fig)
    display(Image(filename=str(path)) if Image is not None else str(path))


# --- 1. two separate observations of the same submission -------------------------------
fig, axes = plt.subplots(1, 2, figsize=(9.2, 2.6))
tiles = [(str(1071), 'observed peak rating', "top of the rating range in this submission's game history"),
         ('890.2', 'stable public score', 'submission of 2026-07-20, status COMPLETE')]
for ax, (value, label, note) in zip(axes, tiles):
    ax.axis('off')
    ax.add_patch(FancyBboxPatch((0.02, 0.05), 0.96, 0.9, boxstyle='round,pad=0.02,rounding_size=0.04',
                                linewidth=1, edgecolor=PAL['grid'], facecolor=PAL['surface']))
    ax.text(0.07, 0.60, value, fontsize=34, color=PAL['series'], va='center')
    ax.text(0.07, 0.33, label, fontsize=11, color=PAL['ink'], va='center')
    ax.text(0.07, 0.18, note, fontsize=9, color=PAL['muted'], va='center')
fig.suptitle('The same agent, two different measurements', fontsize=12, color=PAL['ink'], y=1.04)
fig.text(0.5, -0.10, 'Simulation ratings move as the ladder keeps replaying games: the peak is a moment '
                     'in that history, the public score is where the row settled. Neither is restated as the other.',
         ha='center', fontsize=9, color=PAL['ink2'])
show(fig, 'observations.png')

# --- 2. deck composition ---------------------------------------------------------------
comp = sorted(composition.items(), key=lambda kv: kv[1])
labels = [name for name, _ in comp]
values = [count for _, count in comp]
fig, ax = plt.subplots(figsize=(7.4, 2.9))
bars = ax.barh(labels, values, height=0.62, color=PAL['series'])
for rect, value in zip(bars, values):
    ax.text(rect.get_width() + 0.4, rect.get_y() + rect.get_height() / 2, str(value),
            va='center', fontsize=10, color=PAL['ink'])
ax.set_xlim(0, max(values) * 1.16)
ax.set_title('Deck composition (60 cards)', fontsize=12, color=PAL['ink'], loc='left', pad=10)
ax.tick_params(colors=PAL['ink2'], labelsize=10, length=0)
ax.xaxis.set_visible(False)
for side in ('top', 'right', 'bottom'):
    ax.spines[side].set_visible(False)
ax.spines['left'].set_color(PAL['axis'])
show(fig, 'deck_composition.png')

# --- 3. what is published -------------------------------------------------------------
layers = [('Opponent deck inference', False), ('Leaf evaluator (features + MLP)', False),
          ('Turn planning rollout', False), ('Rule-based skeleton', True)]
fig, ax = plt.subplots(figsize=(7.4, 3.2))
ax.axis('off')
for i, (name, published) in enumerate(layers):
    y = len(layers) - 1 - i
    ax.add_patch(FancyBboxPatch((0.02, y + 0.12), 0.96, 0.74,
                                boxstyle='round,pad=0.01,rounding_size=0.06', linewidth=0,
                                facecolor=PAL['series'] if published else PAL['inactive']))
    ax.text(0.06, y + 0.49, name, fontsize=11, va='center',
            color='white' if published else PAL['ink'])
    ax.text(0.94, y + 0.49, 'in this notebook' if published else 'described only',
            fontsize=9.5, va='center', ha='right',
            color='white' if published else PAL['ink2'])
ax.set_xlim(0, 1)
ax.set_ylim(0, len(layers))
ax.set_title('Agent stack: what this notebook ships', fontsize=12, color=PAL['ink'], loc='left', pad=12)
ax.legend(handles=[Patch(facecolor=PAL['series'], label='published here'),
                   Patch(facecolor=PAL['inactive'], label='described, not included')],
          loc='lower left', bbox_to_anchor=(0.0, -0.16), ncol=2, frameon=False, fontsize=9.5)
show(fig, 'stack.png')


## 8. Digest

In [ ]:
archive_sha = hashlib.sha256(archive.read_bytes()).hexdigest()
print('submission.tar.gz sha256:', archive_sha)
print('main.py          sha256:', EXPECTED_SHA256['main.py'])
print('deck.csv         sha256:', EXPECTED_SHA256['deck.csv'])

for temporary in [WORK / 'main.py', WORK / 'deck.csv', build]:
    try:
        if temporary.is_dir():
            shutil.rmtree(temporary, ignore_errors=True)
        elif temporary.exists():
            temporary.unlink()
    except OSError as exc:  # the engine library stays mapped in-process on some platforms
        print('left in place:', temporary, exc)
assert archive.is_file()
print('outputs: submission.tar.gz + skeleton_visuals/')


## Scope and attribution

- The archive built above is the **skeleton only**. The peak rating of 1071 &mdash; the top of the
  range shown in that submission's game history &mdash; and the public score of 890.2 were recorded
  by an agent that runs search, a learned evaluator and opponent deck inference on top of this
  policy. Those layers are described here, not shipped, and no claim is made that the archive in
  this notebook reproduces those numbers.
- Both figures are observations of the same submission's live rating at different points in its
  history, reported as such. Simulation ratings drift as the ladder replays games; neither number
  is a fixed property of the agent.
- The decklist is a variant of a widely played Alakazam list &mdash; the archetype is public and
  visible in replays. What is mine is the policy: which prompt gets which score, and when the
  engine declines to draw.
- Everything in `main.py` is plain Python against the official `cg` API. There are no external
  dependencies, no network access, and no data files beyond `deck.csv`.
